# MaaS Traffic Generator

Generate varied traffic against MaaS-served models to populate the observability dashboard with rich metrics.

**What this does:**
- Sends diverse prompts (short, medium, long, multi-turn) to multiple models
- Generates enough token volume to show meaningful throughput, latency, and queue metrics
- Runs at a configurable pace so you can observe real-time dashboard updates

## Prerequisites
- A MaaS API key (from Gen AI Studio > API Keys)
- At least one model published to MaaS with a subscription

In [ ]:
import requests
import time
import json
import random
import warnings
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

## Configuration

In [ ]:
API_KEY = "sk-oai-1T4BdIZ1uYF7FCzfd_lEQKny2GX7y5dhNJyjvzUVSjDZ0FEWWgQ4Mq15LrKgA"
MAAS_BASE = "https://maas.apps.ocp.4dgzz.sandbox3174.opentlc.com"

MODELS = [
    {"namespace": "a-rh-department", "name": "maas-qwen35-8b-a3b-fp8"},
    {"namespace": "a-rh-department", "name": "maas-gpt-oss-20b"},
]

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

def chat_url(model):
    return f"{MAAS_BASE}/{model['namespace']}/{model['name']}/v1/chat/completions"

print(f"Base URL: {MAAS_BASE}")
print(f"Models:   {[m['name'] for m in MODELS]}")
print(f"API Key:  {API_KEY[:20]}...")

## Prompt Bank

Varied prompts to generate diverse token patterns — short responses, long completions, reasoning, code generation.

In [ ]:
PROMPTS = {
    "short": [
        ("What is 2+2?", 10),
        ("Say hello.", 10),
        ("Name a color.", 5),
        ("Is the sky blue? Answer yes or no.", 5),
        ("What day comes after Monday?", 10),
    ],
    "medium": [
        ("Explain what Kubernetes is in 3 sentences.", 100),
        ("Write a haiku about machine learning.", 50),
        ("What are the benefits of containerization? Be concise.", 100),
        ("Compare Python and Go for building APIs in 5 bullet points.", 150),
        ("Explain the CAP theorem simply.", 100),
    ],
    "long": [
        ("Write a short story about a robot learning to paint. Include dialogue.", 300),
        ("Explain how neural networks work, from perceptrons to transformers.", 400),
        ("Write a comprehensive guide to setting up a CI/CD pipeline.", 400),
        ("Describe the history of artificial intelligence from the 1950s to today.", 500),
        ("Write a detailed comparison of microservices vs monolithic architecture.", 400),
    ],
    "code": [
        ("Write a Python function that checks if a number is prime.", 150),
        ("Write a bash script to find the largest files in a directory.", 150),
        ("Write a simple REST API in Python using Flask with CRUD operations.", 300),
        ("Write a Python class that implements a binary search tree with insert and search.", 250),
        ("Write a Kubernetes deployment YAML for an nginx pod with 3 replicas.", 150),
    ],
    "reasoning": [
        ("If all roses are flowers and some flowers fade quickly, can we conclude that some roses fade quickly? Explain your reasoning.", 200),
        ("A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left? Think step by step.", 100),
        ("Solve: You have 3 boxes. One contains only apples, one only oranges, one both. Labels are all wrong. You can pick one fruit from one box. How do you label all boxes correctly?", 300),
    ],
}

total_prompts = sum(len(v) for v in PROMPTS.values())
print(f"Prompt bank: {total_prompts} prompts across {len(PROMPTS)} categories")
for cat, prompts in PROMPTS.items():
    print(f"  {cat}: {len(prompts)} prompts")

## Send Request Helper

In [ ]:
def send_request(model, prompt, max_tokens, request_id=0):
    """Send a chat completion request and return structured results."""
    url = chat_url(model)
    start = time.time()
    try:
        resp = requests.post(
            url,
            headers=HEADERS,
            json={
                "model": model["name"],
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_tokens,
            },
            verify=False,
            timeout=120,
        )
        elapsed = time.time() - start
        result = {
            "id": request_id,
            "model": model["name"],
            "status": resp.status_code,
            "elapsed_s": round(elapsed, 2),
            "timestamp": datetime.now().isoformat(),
        }
        if resp.status_code == 200:
            data = resp.json()
            usage = data.get("usage", {})
            result["prompt_tokens"] = usage.get("prompt_tokens", 0)
            result["completion_tokens"] = usage.get("completion_tokens", 0)
            result["total_tokens"] = usage.get("total_tokens", 0)
        elif resp.status_code == 429:
            result["error"] = "RATE_LIMITED"
        else:
            result["error"] = resp.text[:200]
        return result
    except Exception as e:
        return {
            "id": request_id,
            "model": model["name"],
            "status": 0,
            "elapsed_s": round(time.time() - start, 2),
            "error": str(e)[:200],
        }

## Quick Connectivity Check

Verify both models are reachable before starting the full traffic run.

In [ ]:
print("Checking model connectivity...\n")
for model in MODELS:
    r = send_request(model, "Say OK.", 5)
    status = "OK" if r["status"] == 200 else f"FAILED ({r['status']}: {r.get('error', '')})"
    tokens = r.get("total_tokens", "-")
    print(f"  {model['name']}: {status} ({r['elapsed_s']}s, {tokens} tokens)")

print("\nReady to generate traffic!")

## Sequential Traffic — Steady Stream

Send requests one at a time across all prompt categories, alternating between models.
This creates a steady stream of traffic visible in the dashboard's request rate and latency panels.

In [ ]:
NUM_ROUNDS = 3
DELAY = 1.0  # seconds between requests

all_prompts = []
for category, prompts in PROMPTS.items():
    for prompt, max_tokens in prompts:
        all_prompts.append((category, prompt, max_tokens))

results = []
total_tokens = 0
request_num = 0

print(f"Starting {NUM_ROUNDS} rounds x {len(all_prompts)} prompts = {NUM_ROUNDS * len(all_prompts)} requests")
print(f"Estimated duration: ~{NUM_ROUNDS * len(all_prompts) * (DELAY + 2):.0f}s")
print("=" * 80)

for round_num in range(1, NUM_ROUNDS + 1):
    random.shuffle(all_prompts)
    print(f"\n--- Round {round_num}/{NUM_ROUNDS} ---")

    for category, prompt, max_tokens in all_prompts:
        request_num += 1
        model = random.choice(MODELS)
        r = send_request(model, prompt, max_tokens, request_num)
        results.append(r)

        if r["status"] == 200:
            tok = r.get("total_tokens", 0)
            total_tokens += tok
            print(f"  [{request_num:3d}] {r['status']} | {model['name'][:25]:25s} | {category:10s} | {tok:5d} tok | {r['elapsed_s']:5.1f}s")
        elif r["status"] == 429:
            print(f"  [{request_num:3d}] 429 RATE LIMITED | {model['name']} | sleeping 30s...")
            time.sleep(30)
        else:
            print(f"  [{request_num:3d}] {r['status']} ERROR | {model['name']} | {r.get('error', '')[:60]}")

        time.sleep(DELAY)

print("\n" + "=" * 80)
ok = sum(1 for r in results if r["status"] == 200)
err = sum(1 for r in results if r["status"] != 200)
print(f"Done! {ok} OK, {err} errors, {total_tokens:,} total tokens generated")

## Concurrent Traffic — Load Burst

Send multiple requests in parallel to stress the models and generate queue depth metrics.
This is useful for populating the "Request Queue Length" panel in the dashboard.

In [ ]:
CONCURRENT = 5
BURST_REQUESTS = 30

burst_results = []
burst_tokens = 0

print(f"Sending {BURST_REQUESTS} requests with concurrency={CONCURRENT}...")
print("=" * 80)

start_time = time.time()

with ThreadPoolExecutor(max_workers=CONCURRENT) as pool:
    futures = []
    for i in range(BURST_REQUESTS):
        cat = random.choice(list(PROMPTS.keys()))
        prompt, max_tokens = random.choice(PROMPTS[cat])
        model = random.choice(MODELS)
        futures.append(pool.submit(send_request, model, prompt, max_tokens, i + 1))

    for future in as_completed(futures):
        r = future.result()
        burst_results.append(r)
        if r["status"] == 200:
            tok = r.get("total_tokens", 0)
            burst_tokens += tok
            print(f"  [{r['id']:3d}] {r['status']} | {r['model'][:25]:25s} | {tok:5d} tok | {r['elapsed_s']:5.1f}s")
        else:
            print(f"  [{r['id']:3d}] {r['status']} | {r['model'][:25]:25s} | {r.get('error', '')[:40]}")

wall_time = time.time() - start_time
ok = sum(1 for r in burst_results if r["status"] == 200)
print(f"\nBurst complete: {ok}/{BURST_REQUESTS} OK, {burst_tokens:,} tokens in {wall_time:.1f}s")
if ok > 0:
    avg_latency = sum(r["elapsed_s"] for r in burst_results if r["status"] == 200) / ok
    print(f"Avg latency: {avg_latency:.1f}s | Throughput: {burst_tokens / wall_time:.0f} tok/s")

## Summary Statistics

In [ ]:
all_results = results + burst_results
ok_results = [r for r in all_results if r["status"] == 200]

print("=" * 60)
print("TRAFFIC GENERATION SUMMARY")
print("=" * 60)
print(f"Total requests sent:   {len(all_results)}")
print(f"Successful (200):      {len(ok_results)}")
print(f"Rate limited (429):    {sum(1 for r in all_results if r['status'] == 429)}")
print(f"Errors:                {sum(1 for r in all_results if r['status'] not in (200, 429))}")
print()

if ok_results:
    total_tok = sum(r.get('total_tokens', 0) for r in ok_results)
    prompt_tok = sum(r.get('prompt_tokens', 0) for r in ok_results)
    completion_tok = sum(r.get('completion_tokens', 0) for r in ok_results)
    latencies = [r['elapsed_s'] for r in ok_results]

    print(f"Total tokens:          {total_tok:,}")
    print(f"  Prompt tokens:       {prompt_tok:,}")
    print(f"  Completion tokens:   {completion_tok:,}")
    print()
    print(f"Latency (min/avg/max): {min(latencies):.1f}s / {sum(latencies)/len(latencies):.1f}s / {max(latencies):.1f}s")
    print()

    from collections import Counter
    model_counts = Counter(r['model'] for r in ok_results)
    print("Per-model breakdown:")
    for model, count in model_counts.most_common():
        model_tok = sum(r.get('total_tokens', 0) for r in ok_results if r['model'] == model)
        print(f"  {model}: {count} requests, {model_tok:,} tokens")

print("\nDashboard should now show metrics at:")
print("  RHOAI Dashboard > Observe & Monitor > Dashboard")